## RQ1 and RQ2: targeted vs random extra training, frozen vs fine-tuned embeddings

- **RQ1**: does extra training on hard sentences beat the same amount of extra training on random sentences?
- **RQ2**: does fine-tuning the embeddings help, compared with keeping FinBERT's embeddings frozen?

Same configuration and method as experiment 7 (`03_finbert_cnn_hard_sets.ipynb`), shared through `utils/finbert_cnn.py`:

1. Baseline CNN on FinBERT (`yiyanghkust/finbert-pretrain`) word-piece embeddings, trained on the full training set.
2. A copy of the baseline is trained further on the full training set with a subset **up-weighted** (weight 1.5, Adam 1e-4, early stopping on val macro F1):
   - **targeted**: the P20 hard training set
   - **random control**: a random subset of the same size, drawn with the same seed
3. Score on the validation set (overall / hard / non-hard, macro F1). Hard subsets are the P20 subsets saved by notebook 03 in `results/hard_samples/`.

| | frozen embeddings | fine-tuned embeddings |
|---|---|---|
| targeted | #2 | #4 |
| random control | #3 | #5 |

The only difference between targeted and random control is *which* sentences get up-weighted. Test set is not used.

In [ ]:
import os
os.environ["HF_HOME"] = os.path.abspath("../.cache/huggingface")
os.environ["HF_HUB_CACHE"] = os.path.abspath("../.cache/huggingface/hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from tensorflow import keras
from tqdm import tqdm

from utils.finbert_cnn import BASELINE_SEED, SEEDS, load_finbert_inputs, load_hard_selection, train_baseline
from utils.preprocessing import LABELS, encode_labels
from utils.random_control import (
    HARD_PERCENTILE, embedding_name, run_random_control, score_predictions, summarize_seeds,
    to_metrics_rows, update_metrics_csv,
)
from utils.random_control import CONDITION as RANDOM
from utils.targeted import CONDITION as TARGETED, run_targeted

# Same seed as experiment 7: the baseline comes out identical.
keras.utils.set_random_seed(BASELINE_SEED)

### Data (FinBERT word-piece tokenization, max length 64)

In [ ]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
y_train = encode_labels(train_df.label)
y_val = encode_labels(val_df.label)

X_train, X_val, embedding_matrix = load_finbert_inputs(train_df, val_df)
data = {"X_train": X_train, "y_train": y_train, "X_val": X_val, "y_val": y_val}
print(X_train.shape, X_val.shape, embedding_matrix.shape)

### Hard subsets (P20, saved by notebook 03)
- **Training** hard set: bottom 20% margin of out-of-fold CNN predictions. Targeted up-weights these; random control up-weights the same *number* of random sentences.
- **Validation** hard set: the training cutoff applied to the frozen baseline's validation predictions.

Re-training the out-of-fold models does not give bit-identical sets, so loading them is the only way to score every experiment on exactly the same sentences.

In [ ]:
hard_train_idx, val_hard_idx = load_hard_selection(HARD_PERCENTILE)
print(f"train hard (up-weighted): {len(hard_train_idx)} / {len(y_train)}   |   val hard: {len(val_hard_idx)} / {len(y_val)}")

### Baselines: frozen (#2, #3) and fine-tuned embeddings (#4, #5)

In [ ]:
baselines = {}
for trainable in [False, True]:
    keras.utils.set_random_seed(BASELINE_SEED)
    baselines[trainable] = train_baseline(embedding_matrix, X_train, y_train, X_val, y_val, trainable=trainable)

### Run: targeted and random control, frozen and fine-tuned, all seeds

In [ ]:
rows = []
for trainable, baseline in baselines.items():
    for seed in tqdm(SEEDS, desc=embedding_name(trainable)):
        rows.append({"condition": TARGETED, **run_targeted(baseline, data, trainable, seed, hard_train_idx, val_hard_idx)})
        rows.append({"condition": RANDOM, **run_random_control(baseline, data, trainable, seed, hard_train_idx, val_hard_idx)})
seed_df = pd.DataFrame(rows)
seed_df

### Check: the frozen runs reproduce experiment 7 (P20)

In [ ]:
exp7 = pd.read_csv("../results/rq1_rq2/percentile_seeds.csv")
exp7 = exp7[exp7.percentile == HARD_PERCENTILE].dropna(subset=["seed"]).astype({"seed": int})
ours = seed_df[seed_df.embedding_config == embedding_name(False)]
check = ours.merge(exp7, left_on=["condition", "seed"], right_on=["strategy", "seed"])
pairs = [("f1_overall", "macro_f1"), ("f1_hard", "hard_macro_f1"),
         ("accuracy_overall", "accuracy"), ("accuracy_hard", "hard_accuracy")]
print("identical to experiment 7:", len(check) == len(ours) and all(np.allclose(check[a], check[b]) for a, b in pairs))
check[["condition", "seed"] + [c for pair in pairs for c in pair]]

### Results (mean and standard deviation over seeds)

In [ ]:
summaries = {condition: summarize_seeds(group) for condition, group in seed_df.groupby("condition")}
for condition, (mean, std) in summaries.items():
    print(condition)
    display(mean.round(3))
    display(std.round(3))

### Save
- per-seed details: `results/rq1_rq2/seeds.csv`
- baseline validation predictions: `results/predictions/baseline_finbert_cnn_{frozen,finetuned}.csv`
- one row per experiment in the shared `metrics.csv`: the two baselines and the four averaged arms

In [ ]:
seed_df.to_csv("../results/rq1_rq2/seeds.csv", index=False)

baseline_scores = {}
for trainable, baseline in baselines.items():
    probs = baseline.predict(X_val, batch_size=32, verbose=0)
    out = val_df[["sentence", "label"]].copy()
    out[["p_negative", "p_neutral", "p_positive"]] = probs
    out["p_true"] = probs[np.arange(len(val_df)), y_val]
    out["pred"] = [LABELS[i] for i in probs.argmax(axis=1)]
    out.to_csv(f"../results/predictions/baseline_finbert_cnn_{embedding_name(trainable).split('_')[1]}.csv", index=False)
    baseline_scores[embedding_name(trainable)] = score_predictions(y_val, probs, val_hard_idx)

baseline_rows = to_metrics_rows(pd.DataFrame(baseline_scores).T.rename_axis("embedding_config"), "baseline")
arm_rows = [to_metrics_rows(mean, condition) for condition, (mean, _) in summaries.items()]
update_metrics_csv("../results/metrics.csv", pd.concat([baseline_rows, *arm_rows], ignore_index=True))